In [19]:
import gymnasium as gym
import collections
import wandb
from typing import Dict, Tuple

In [20]:
from kaggle_secrets import UserSecretsClient
user_secrets = UserSecretsClient()
secret_value_0 = user_secrets.get_secret("WANDB_API_KEY")

In [21]:
wandb.login(key=secret_value_0)

wandb: WARNING If you're specifying your api key in code, ensure this code is not shared publicly.
wandb: WARNING Consider setting the WANDB_API_KEY environment variable, or running `wandb login` from the command line.
wandb: WARNING [wandb.login()] Changing session credentials to explicit value for https://api.wandb.ai.
wandb: Appending key for api.wandb.ai to your netrc file: /root/.netrc


True

In [22]:
CONFIG = {
    "env_name": "FrozenLake-v1",
    "gamma": 0.9,
    "test_episodes": 20,
    "random_steps_per_iter": 100,
    "solve_threshold": 0.80}

In [23]:
class Agent:
    def __init__(self, env_name: str, gamma: float):
        self.env = gym.make(env_name)
        self.state, _ = self.env.reset()
        self.gamma = gamma
        self.rewards: Dict[Tuple[int, int, int], float] = collections.defaultdict(float)
        self.transits: Dict[Tuple[int, int], collections.Counter] = collections.defaultdict(collections.Counter)
        self.values: Dict[int, float] = collections.defaultdict(float)

    def play_n_random_steps(self, count: int):
        for _ in range(count):
            action = self.env.action_space.sample()
            new_state, reward, terminated, truncated, _ = self.env.step(action)
            
            self.rewards[(self.state, action, new_state)] = float(reward)
            self.transits[(self.state, action)][new_state] += 1
            
            if terminated or truncated:
                self.state, _ = self.env.reset()
            else:
                self.state = new_state

    def calc_action_value(self, state: int, action: int) -> float:
        target_counts = self.transits[(state, action)]
        total = sum(target_counts.values())
        if total == 0:
            return 0.0
        
        action_value = 0.0
        for tgt_state, count in target_counts.items():
            reward = self.rewards[(state, action, tgt_state)]
            # Bellman Equation: P(s'|s,a) * (R + gamma * V(s'))
            action_value += (count / total) * (reward + self.gamma * self.values[tgt_state])
        return action_value

    def select_action(self, state: int) -> int:
        best_action, best_value = 0, -1.0
        for action in range(self.env.action_space.n):
            action_value = self.calc_action_value(state, action)
            if action_value > best_value:
                best_value = action_value
                best_action = action
        return best_action

    def play_episode(self, env: gym.Env) -> float:
        total_reward = 0.0
        state, _ = env.reset()
        while True:
            action = self.select_action(state)
            new_state, reward, terminated, truncated, _ = env.step(action)
            
            self.rewards[(state, action, new_state)] = float(reward)
            self.transits[(state, action)][new_state] += 1
            
            total_reward += float(reward)
            if terminated or truncated:
                break
            state = new_state
        return total_reward

    def value_iteration(self):
        for state in range(self.env.observation_space.n):
            action_values = [
                self.calc_action_value(state, action)
                for action in range(self.env.action_space.n)
            ]
            if action_values:
                self.values[state] = max(action_values)

In [24]:
run = wandb.init(
        project="frozen-lake-v-iteration",
        config=CONFIG,
        monitor_gym=True
    )

In [25]:
test_env = gym.make(CONFIG["env_name"])
agent = Agent(CONFIG["env_name"], CONFIG["gamma"])

iter_no = 0
best_reward = 0.0

In [26]:
try:
        while True:
            iter_no += 1
            
            # 1. Gather Experience
            agent.play_n_random_steps(CONFIG["random_steps_per_iter"])
            
            # 2. Update Values (Dynamic Programming)
            agent.value_iteration()

            # 3. Evaluation
            total_test_reward = 0.0
            for _ in range(CONFIG["test_episodes"]):
                total_test_reward += agent.play_episode(test_env)
            
            avg_reward = total_test_reward / CONFIG["test_episodes"]
            
            # Log to WandB
            wandb.log({
                "iteration": iter_no,
                "test_reward": avg_reward,
                "best_reward": max(best_reward, avg_reward)
            })

            if avg_reward > best_reward:
                print(f"Iter {iter_no}: Best reward updated {best_reward:.3f} -> {avg_reward:.3f}")
                best_reward = avg_reward
            
            if avg_reward > CONFIG["solve_threshold"]:
                print(f"Solved in {iter_no} iterations!")
                break
                
finally:
        run.finish()
        test_env.close()

Iter 4: Best reward updated 0.000 -> 0.050
Iter 6: Best reward updated 0.050 -> 0.150
Iter 7: Best reward updated 0.150 -> 0.200
Iter 8: Best reward updated 0.200 -> 0.250
Iter 9: Best reward updated 0.250 -> 0.300
Iter 13: Best reward updated 0.300 -> 0.350
Iter 14: Best reward updated 0.350 -> 0.450
Iter 20: Best reward updated 0.450 -> 0.600
Iter 28: Best reward updated 0.600 -> 0.650
Iter 36: Best reward updated 0.650 -> 0.700
Iter 37: Best reward updated 0.700 -> 0.800
Iter 43: Best reward updated 0.800 -> 0.850
Solved in 43 iterations!


best_reward,▁▁▁▁▁▂▃▃▃▃▃▃▄▅▅▅▅▅▆▆▆▆▆▆▆▆▆▆▆▆▆▆▆▇██████
iteration,▁▁▁▁▂▂▂▂▂▃▃▃▃▃▃▄▄▄▄▄▅▅▅▅▅▅▆▆▆▆▆▇▇▇▇▇▇▇██
test_reward,▁▁▁▁▁▂▃▃▃▂▃▃▄▃▅▄▄▄▆▅▅▃▄▃▄▄▅▃▃▄▅▄▆▇█▇▇▇▆█
best_reward,0.85
iteration,43
test_reward,0.85
